# 04: Title model: mT5-small (embeddings frozen)

**Task:** chapter text (the ASR sentences inside the chapter) → chapter title, in the same language as the text.

**Why frozen embeddings:** measured on the 1060 (research §5), full fine-tuning OOMs even at batch 1. Freezing the shared embedding and the LM head leaves ~44 M trainable params and peaks at ~1.8 GB (batch 4 × 256 tokens).

**Pairs:**
- from the `train` split: English capped at `EN_PAIRS`, and **all** Spanish **oversampled ×`ES_OVERSAMPLE`**, because only ~900 Spanish pairs exist (see the 01 log);
- validation on `val` (en) and `es_val` (es), with ground-truth boundaries. This is VidChapters' "title generation with GT boundaries" task.

**Reference point:** an extractive baseline (`lead-8`, the first 8 words of the chapter). The paper's number (Vid2Seq speech-only: CIDEr 120.5, B1 23.5) was measured on a different test set with PTB tokenisation, so it is **not** directly comparable.

In [1]:
import json, random, time
from pathlib import Path
import pandas as pd, torch
from dotenv import load_dotenv
load_dotenv('../.env')
from lib import data as d, vidchapters as vc, titles as T

ROOT = d.data_root(); RUN = Path('runs/2026-09-28-titles'); RUN.mkdir(parents=True, exist_ok=True)
CKPT = ROOT / 'checkpoints' / 'titles'
EN_PAIRS, VAL_EN, VAL_ES = 60000, 500, 300
ES_OVERSAMPLE = 8  # only ~900 Spanish pairs exist (01_data); keep mT5 answering in Spanish
CFG = T.TitleConfig(max_input_tokens=256, batch_size=4, grad_accum=4, lr=1e-3, epochs=1, eval_every=2000)
DEVICE = 'cuda'
rng = random.Random(0)
def pairs_for(split): return T.make_pairs(list(vc.load_jsonl(ROOT / f'interim/{split}.jsonl')))
train_all = pairs_for('train')
en = [p for p in train_all if p['lang'] == 'en']; es = [p for p in train_all if p['lang'] == 'es']
rng.shuffle(en)
train_pairs = en[:EN_PAIRS] + es * ES_OVERSAMPLE
val_en = rng.sample(pairs_for('val'), VAL_EN); val_es = pairs_for('es_val'); val_es = rng.sample(val_es, min(VAL_ES, len(val_es)))
print(f'train pairs: {len(train_pairs)} (en {min(len(en), EN_PAIRS)} of {len(en)}, es {len(es)} x{ES_OVERSAMPLE}); val en {len(val_en)}, val es {len(val_es)}')
pd.DataFrame(train_pairs[:5])[['lang', 'title', 'text']].assign(text=lambda x: x.text.str[:120])

/home/pablo/projects/new_youtube_suite/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train pairs: 66760 (en 60000 of 86704, es 845 x8); val en 500, val es 117


,lang,title,text
0,en,Importing Audio,Now let's add some audio to this file. Adding ...
1,en,Setup Discord Application / Bot,"All right, so we need to go into our Discord d..."
2,en,Job hunting. 1:02:27 My book (link below). 1:1...,"No, it's funny as a therapist, sometimes peopl..."
3,en,Hyperkin RetroN 5,Next up is the Hyperkin Retron 5. And for year...
4,en,Is intel investing to advance the chips or sim...,accelerate the growth of use of their chips? L...


## 1. Extractive reference: the chapter's first 8 words

In [2]:
lead = lambda ps: [' '.join(p['text'].split()[:8]) for p in ps]
ref_scores = {name: T.score([p['title'] for p in ps], lead(ps)) for name, ps in (('val_en', val_en), ('val_es', val_es))}
pd.DataFrame(ref_scores).T.round(2)

,B1,B4,CIDEr
val_en,8.04,2.40,30.22
val_es,7.88,2.13,32.75


## 2. Fine-tune (checkpoint = best CIDEr on validation, en + es mixed)

In [3]:
tok, model = T.load_model(DEVICE)
t0 = time.time()
history = T.train_titles(tok, model, train_pairs, val_en + val_es, CFG, CKPT, DEVICE)
print(f'{(time.time()-t0)/60:.0f} min')
pd.DataFrame([{k: v for k, v in h.items() if k != 'samples'} for h in history])

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 24318.48it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{"step": 2000, "sec": 1572, "trainable_params": 44062080, "B1": 14.021393466315693, "B4": 4.913398742094853, "CIDEr": 61.78245198605312}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.68s/it]

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.68s/it]

56 min


,step,sec,trainable_params,B1,B4,CIDEr
0,2000,1572,44062080,14.021393,4.913399,61.782452
1,4000,3142,44062080,15.054637,5.181089,63.675803
2,4173,3333,44062080,14.045416,5.005917,64.721814


## 3. Best checkpoint per language, next to the extractive reference

In [4]:
tok, model = T.load_model(DEVICE, CKPT / 'best')
rows, examples = [], {}
for name, ps in (('val_en', val_en), ('val_es', val_es)):
    hyps = T.generate(tok, model, [p['text'] for p in ps], DEVICE, CFG)
    rows.append(dict(set=name, method='mt5-small', **T.score([p['title'] for p in ps], hyps)))
    rows.append(dict(set=name, method='lead-8', **ref_scores[name]))
    examples[name] = pd.DataFrame(dict(reference=[p['title'] for p in ps], generated=hyps, text=[p['text'][:100] for p in ps]))
pd.DataFrame(rows).round(2)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5786.51it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,set,method,B1,B4,CIDEr
0,val_en,mt5-small,14.38,4.79,67.88
1,val_en,lead-8,8.04,2.40,30.22
2,val_es,mt5-small,12.31,6.11,49.99
3,val_es,lead-8,7.88,2.13,32.75


In [5]:
examples['val_en'].head(15)

,reference,generated,text
0,How is the Vision Fund II raise coming along,What’s going on with Vision Fund 2,What's going on with Vision Fund 2? I hear the...
1,Posters for Sale,Closing Thoughts,Check out this cool poster that I got from Chr...
2,Check Your Solar Is Working Properly – With Th...,What forecasts the output of solar farms using...,He's alright by me because he lives in a carav...
3,"Bill: If Kavanaugh did do it, he’s now a liar",What is Mark joseph stern,"I think first, we would need to establish whet..."
4,Segregation & Brutality at the training school,What does the training school put you through,Like what does the training school put you thr...
5,Making The Pivot To Functional Medicine,What is my North Star,And it's metabolic dysfunction. It's inflammat...
6,You Satisfy My Soul - (Laura Park) 6495147,You satisfy my soul,And you satisfy my soul. You satisfy my soul. ...
7,"When you were writing ""Sideways"" did you think...","Why did you think that you had no idea, consid...","But did you think, writing it, obviously, I'm ..."
8,Road Salt in Lakes,Road Salt Runoff,"Hello, and thanks for joining us. A recent stu..."
9,Wisk Power Scoop Detergent,Greasy stains and stains,We're at this greasy spoon to make a point abo...


In [6]:
examples['val_es'].head(15)

,reference,generated,text
0,Hasta El Amanecer,Cuando llego,Y como tú te llamas yo no sé De dónde llegaste...
1,Infrastructura como código,Utilizan normalmente templates de CloudFormation,Utilizan normalmente templates de CloudFormati...
2,Picky,Cuando nos vamos al final,"Le digo, le llame, dice goodbye Le digo, nena,..."
3,Andas En Mi Cabeza,Contigo me casé y por siempre te amé,Si te vas quedar en un dolor que jamás conocí ...
4,Livin'in Jam,DJ Defeating Sam,"Mmm, ok mi despierto, buenos días Uy Alright, ..."
5,AWS EC2 - Virtualización en AWS,EC2: ¿Qué es EC2? ¿Qué es EC2? ¿Qué es EC2? ¿Q...,"¿Qué es EC2? Y EC2 es, en realidad, para la ge..."
6,Guarda los Estornudos para Tí Keep Your Sneeze...,Cuando vas a estornudar,Todos los días compartimos muchas cosas con nu...
7,Curitas,Yo Te Extraño,Yo te extraño Yo te extraño Me extraño a mí Es...
8,Traidora,Cuando puedo estar a mi corazón,Yo solo quiero darte amor Solo quiero estar ju...
9,Paso 6: Lleve la impresora al servicio técnico,Chapter 5: „Enviar una impresora al servicio t...,Si ha seguido todos los pasos y sigue teniendo...


In [7]:
json.dump(dict(cfg=CFG.__dict__, en_pairs=EN_PAIRS, es_oversample=ES_OVERSAMPLE, n_train=len(train_pairs), n_val_en=len(val_en), n_val_es=len(val_es)),
          open(RUN / 'config.json', 'w'), indent=1)
json.dump(dict(results=rows, history=[{k: v for k, v in h.items() if k != 'samples'} for h in history]),
          open(RUN / 'metrics.json', 'w'), indent=1)
for k, df in examples.items(): df.to_csv(RUN / f'examples_{k}.csv', index=False)
print('saved to', RUN)

saved to runs/2026-09-28-titles
